In [1]:
# --- Create Version 1 ---
v1_text = """You are ShopEasy Support Bot. Answer ONLY from the provided context.
If the answer is not in the context, say: "I don't have that information."
Keep replies under 3 sentences. Use polite Indian English."""


v2_text = """You are ShopEasy Support Bot. Answer ONLY from the provided context.
If the answer is not in the context, say: "I don't have that information."
Keep replies under 3 sentences. Use polite Indian English.
Always end with: "Need anything else? Reply with your order ID."""


In [8]:
with open("data/prompt-versions/version_1.txt","w") as file:
    file.write(v1_text)

with open("data/prompt-versions/verison_2.txt","w") as file:
    file.write(v2_text)

In [ ]:
import json
REGISTRY = {
    "support_agent": {
        "v1": {
            "prompt_path":"data/prompt-versions/version_1.txt",
            "config": {"model": "llama-3.3-70b-versatile", "temperature": 0.0, "max_tokens": 512},
            "tools": ["lookup_order"],  # Tools this version is allowed to use
        },
        "v2": {
            "prompt_path": "data/prompt-versions/verison_2.txt",
            "config": {"model": "llama-3.3-70b-versatile", "temperature": 0.0, "max_tokens": 512},
            "tools": ["lookup_order"],
        },
    }
}

with open("data/config/config_1.json","w") as file :
    json.dump(REGISTRY,file)

In [31]:
with open("data/config/config_1.json","r") as file:
    details=json.load(file)

display(details)

model=details["support_agent"]["v1"]["config"]["model"]

temprature=details["support_agent"]["v1"]["config"]['temperature']
system_prompt_path=details["support_agent"]["v1"]["prompt_path"]
with open(system_prompt_path,"r") as file:
    system_prompt=file.read()


{'support_agent': {'v1': {'prompt_path': 'data/prompt-versions/version_1.txt',
   'config': {'model': 'llama-3.3-70b-versatile',
    'temperature': 0.0,
    'max_tokens': 512},
   'tools': ['lookup_order']},
  'v2': {'prompt_path': 'data/prompt-versions/verison_2.txt',
   'config': {'model': 'llama-3.3-70b-versatile',
    'temperature': 0.0,
    'max_tokens': 512},
   'tools': ['lookup_order']}}}

In [40]:
import os
from dotenv import load_dotenv
load_dotenv()
import groq
from groq import Groq
api_key=os.getenv("api_key")
client=Groq(api_key=api_key)
from tenacity import retry_if_exception_type
from tenacity import retry, wait_exponential, stop_after_attempt

question="hey how the hell are you"
def log_groq_retry(retry_state):
    print(f"⚠️ Groq API Rate Limit hit. Retrying in {retry_state.idle_for:.1f} seconds... (Attempt {retry_state.attempt_number})")

@retry(
    retry=retry_if_exception_type(groq.RateLimitError),
    wait=wait_exponential(multiplier=1, min=1, max=10),
    stop=stop_after_attempt(4),
    before_sleep=log_groq_retry
)
def answer(model,system_prompt,question,temprature):
    response=client.chat.completions.create(
        model=model,
        messages=[{
            "role":"sytem","content":system_prompt,
            "role":"user","content":question
        }],
        temperature=temprature
    )

    return (response.choices[0].message.content)
answer(model,system_prompt,question,temprature)

"I'm just a language model, so I don't have feelings or emotions like humans do, but I'm functioning properly and ready to help with any questions or topics you'd like to discuss. How about you? How's your day going so far?"